## 介绍 

本课内容包括： 
- 什么是函数调用及其使用场景 
- 如何使用 OpenAI 创建函数调用 
- 如何将函数调用集成到应用程序中 

## 学习目标 

完成本课后，您将了解并掌握： 

- 使用函数调用的目的 
- 使用 OpenAI 服务设置函数调用 
- 设计适合您应用场景的高效函数调用 


## 理解函数调用 

在本课中，我们希望为我们的教育初创公司构建一个功能，允许用户使用聊天机器人查找技术课程。我们将推荐适合他们技能水平、当前职位和感兴趣技术的课程。 

为此，我们将结合使用： 
 - `OpenAI` 为用户创建聊天体验
 - `Microsoft Learn Catalog API` 帮助用户根据请求查找课程 
 - `函数调用` 将用户的查询传递给函数以发起 API 请求。 

要开始，让我们看看为什么我们首先要使用函数调用： 


### 为什么使用函数调用

如果您完成了本课程中的其他任何课程，您可能已经理解了使用大型语言模型（LLM）的强大功能。希望您也能看到它们的一些局限性。

函数调用是OpenAI服务的一项功能，旨在解决以下挑战：

响应格式不一致：
- 在函数调用之前，大型语言模型的响应是无结构且不一致的。开发人员必须编写复杂的验证代码来处理输出中的每种变体。

与外部数据集成有限：
- 在此功能之前，很难将应用程序其他部分的数据整合到聊天上下文中。

通过标准化响应格式并实现与外部数据的无缝集成，函数调用简化了开发流程，减少了额外验证逻辑的需求。

用户无法获得诸如“斯德哥尔摩当前天气如何？”之类的答案。这是因为模型被限制在数据训练的时间点。

让我们看看下面的示例，来说明这个问题：

假设我们想创建一个学生数据的数据库，以便向他们推荐合适的课程。下面有两个学生描述，它们的数据信息非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我们想把这个发送给一个大型语言模型（LLM）来解析数据。这可以在我们的应用程序中用于将数据发送给API或存储在数据库中。 

让我们创建两个相同的提示，告诉LLM我们感兴趣的信息： 


我们想把这个发送给大型语言模型（LLM），让它解析对我们产品重要的部分。因此，我们可以创建两个相同的提示来指示LLM： 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


创建这两个提示后，我们将使用 `client.responses.create` 将它们发送给 LLM。我们将提示存储在 `input` 变量中，并将角色分配为 `user`。这是为了模拟用户向聊天机器人发送消息的情景。



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-5-mini"


: 

现在我们可以向 LLM 发送这两个请求并检查我们收到的响应。


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



尽管提示相同且描述类似，我们仍然可能获得不同格式的 `Grades` 属性。

如果多次运行上面的单元格，格式可能是 `3.7` 或 `3.7 GPA`。

这是因为大语言模型接受的是无结构化数据（以书面提示的形式），并返回同样无结构化的数据。我们需要一个结构化的格式，以便在存储或使用这些数据时知道预期内容。

通过使用函数调用，我们可以确保收到结构化的数据。使用函数调用时，LLM 实际上并不调用或运行任何函数。相反，我们为 LLM 创建一个响应结构，然后利用这些结构化的响应来确定在我们的应用程序中应该运行哪个函数。
 


![函数调用流程图](../../../../translated_images/zh-CN/Function-Flow.083875364af4f4bb.webp)


然后我们可以获取函数返回的内容，并将其发送回大型语言模型。大型语言模型随后将使用自然语言回应，以回答用户的查询。


### 使用函数调用的用例 

<strong>调用外部工具</strong> 
聊天机器人擅长为用户提供问题答案。通过使用函数调用，聊天机器人可以使用用户的消息来完成某些任务。例如，学生可以请求聊天机器人“给我的导师发送电子邮件，说我需要更多关于这门课的帮助”。这可以调用一个函数 `send_email(to: string, body: string)`


**创建API或数据库查询**
用户可以使用自然语言查找信息，这些内容会被转换为格式化的查询或API请求。例如，老师可以请求“完成上次作业的学生是谁”，这可以调用一个名为 `get_completed(student_name: string, assignment: int, current_status: string)` 的函数


<strong>创建结构化数据</strong>
用户可以处理一段文本或CSV，并使用大语言模型从中提取重要信息。例如，学生可以把一篇关于和平协议的维基百科文章转换成AI闪卡。这可以通过使用一个名为 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 的函数来完成


## 2. 创建你的第一个函数调用 

创建函数调用的过程包括三个主要步骤： 
1. 使用包含你的函数列表和用户信息的请求调用聊天补全 API 
2. 读取模型的响应以执行操作，即执行函数或 API 调用 
3. 使用你的函数响应，再次调用聊天补全 API，利用该信息生成对用户的回复。 


![函数调用流程](../../../../translated_images/zh-CN/LLM-Flow.3285ed8caf4796d7.webp)


### 函数调用的要素

#### 用户输入

第一步是创建用户消息。这可以通过获取文本输入的值动态分配，或者您也可以在这里分配一个值。如果这是您第一次使用 Chat Completions API，我们需要定义消息的 `role` 和 `content`。

`role` 可以是 `system`（创建规则）、`assistant`（模型）或 `user`（终端用户）。对于函数调用，我们将其分配为 `user` 并给出一个示例问题。


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 创建函数。

接下来我们将定义一个函数及其参数。这里我们只使用一个名为 `search_courses` 的函数，但你可以创建多个函数。

<strong>重要</strong> ：函数包含在发送给大型语言模型的系统消息中，并计入你可用的令牌数。


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定义</strong> 

函数定义结构有多个层级，每个层级都有其自身的属性。以下是嵌套结构的详细说明：

**顶层函数属性：**

`name` - 我们希望调用的函数名称。 

`description` - 这是函数如何工作的描述。这里需要具体且清晰 

`parameters` - 模型在响应中需要生成的值和格式列表 

**参数对象属性：**

`type` - 参数对象的数据类型（通常为 "object"）

`properties` - 模型将用于响应的具体值列表 

**单个参数属性：**

`name` - 由属性键隐式定义（例如 "role", "product", "level"）

`type` - 该特定参数的数据类型（例如 "string", "number", "boolean"） 

`description` - 该特定参数的描述 

**可选属性：**

`required` - 一个数组，列出完成函数调用所需的必需参数 


### 调用函数
定义函数后，我们需要在调用 Chat Completion API 时将其包含在内。我们通过在请求中添加 `functions` 来完成这一步。在本例中为 `functions=functions`。

还有一个选项是将 `function_call` 设置为 `auto`。这意味着我们将让大语言模型根据用户消息决定调用哪个函数，而不是我们自己指定。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


现在让我们来看一下响应，看看它是如何格式化的：

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以看到函数的名称被调用了，并且从用户的信息中，LLM能够找到适合函数参数的数据。


## 3. 将函数调用集成到应用程序中  


在我们测试过LLM格式化响应之后，现在我们可以将其集成到应用程序中了。  

### 管理流程  

要将其集成到我们的应用程序中，让我们执行以下步骤：  

首先，让我们调用OpenAI服务并将消息存储在名为`response_message`的变量中。  


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


现在我们将定义一个函数，该函数将调用 Microsoft Learn API 以获取课程列表： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作为最佳实践，我们将接着查看模型是否想要调用某个函数。之后，我们将创建一个可用的函数，并将其与被调用的函数进行匹配。
然后，我们将把函数的参数映射到来自 LLM 的参数。

最后，我们将附加函数调用消息和 `search_courses` 消息返回的值。这为 LLM 提供了所有它需要的信息，
以便用自然语言回应用户。


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


现在我们将发送更新后的消息给LLM，这样我们就可以收到自然语言响应，而不是API JSON格式的响应。 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 代码挑战 

做得好！为了继续学习 OpenAI 函数调用，你可以构建：https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 更多函数参数，可能帮助学习者找到更多课程。你可以在这里找到可用的 API 参数： 
 - 创建另一个函数调用，从学习者那里获取更多信息，比如他们的母语 
 - 当函数调用和/或 API 调用未返回任何合适的课程时，创建错误处理 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
